# Model 2 V2: Fine-tuned DistilBERT — Longer Training (batch=64, 15 epochs)

**Architecture:** Same as V1 — DistilBERT [CLS] pooling + regression head  
**Change from V1:** batch_size=64, epochs=15, patience=3, warmup_steps=500  
**Hypothesis:** V1 val MAE was still decreasing at epoch 5 ($47.42) — more training should improve further

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

In [ ]:
from pricer.items import Item
from pricer.distilbert_model_v2 import DistilBERTRunnerV2
from pricer.evaluator import evaluate, plot_training_history

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_full")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Setup Model

Same DistilBERT architecture as V1, batch_size=64.

In [ ]:
runner = DistilBERTRunnerV2(train, val[:1000])
runner.setup(batch_size=64)

## 3. Train

Max 15 epochs, early stopping patience=3, linear warmup 500 steps.

In [ ]:
history = runner.train(epochs=15, patience=3, warmup_steps=500)

## 4. Training History

In [ ]:
plot_training_history(history, title="DistilBERT V2 (CLS, batch=64, 15 epochs)")

## 5. Evaluate on 200 Test Samples

In [ ]:
evaluate(runner.inference, test)

## 6. Save Model Weights

In [ ]:
runner.save("distilbert_model_v2.pth")
print("Saved to distilbert_model_v2.pth")

## 7. Sanity Check

In [ ]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred:.2f}")
print(f"Error:   ${abs(pred - sample.price):.2f}")